In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

¡CONECTADO! Título de la página: Google


# Vamos a probar con un código genérico para cualquier página y vamos cambiando las etiquetas y las busquedas

In [36]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://energia.gob.cl/noticias/antofagasta/con-amplia-participacion-ciudadana-avanza-la-construccion-del-plan-de-accion-de-hidrogeno-verde-2023-2030" 

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = ".views-row" 
SELECTOR_DATO_A     = "span.views-field-title"
SELECTOR_DATO_B     = "div.views-field-created"
# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) 

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Guardamos el texto sin borrar las letras
            valor_final = columna_b.strip()

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_final,
                "status": 1.0 
            })
        except:
            # Si una fila falla, sigue con la siguiente
            continue
# ==========================================================
# 4. SALIDA DE DATOS (Dentro del try principal)
# ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df)
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

except Exception as e:
    print(f" Error general: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
 Se encontraron 7 registros potenciales.
 No se capturaron datos. Revisar los selectores CSS.
 Navegador cerrado.


In [33]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# 1. MOTOR DE NAVEGACIÓN
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. CONFIGURACIÓN
URL_OBJETIVO = "https://energia.gob.cl"
SELECTOR_CONTENEDOR = ".region-content" # Selector general para noticias
SELECTOR_DATO_A = "h1.title"            # El título
SELECTOR_DATO_B = ".field-name-body"    # El texto

# 3. EJECUCIÓN
driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)

    # Extraemos los datos
    tit = driver.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
    txt = driver.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

    # Guardamos la info (sin borrar letras)
    dataset_final.append({
        "variable_nombre": "Titulo",
        "variable_valor": tit,
        "status": 1.0
    })
    dataset_final.append({
        "variable_nombre": "Contenido",
        "variable_valor": txt[:500], # Primeros 500 caracteres
        "status": 1.0
    })

    # 4. SALIDA (Alineado con el código de arriba)
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False)
        print("¡LOGRADO! Archivo generado con éxito.")
        print(df)
    else:
        print("No se capturaron datos.")

except Exception as e:
    print(f"Error: {e}")

finally:
    driver.quit()
    print("Navegador cerrado.")


Conectando...
Error: Message: no such element: Unable to locate element: {"method":"css selector","selector":"h1.title"}
  (Session info: chrome=146.0.7680.164); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
#0 0x602e00799a6a <unknown>
#1 0x602e001a8ab5 <unknown>
#2 0x602e001fb676 <unknown>
#3 0x602e001fb8b1 <unknown>
#4 0x602e00246614 <unknown>
#5 0x602e002437b6 <unknown>
#6 0x602e001eecbf <unknown>
#7 0x602e001efa81 <unknown>
#8 0x602e0075fa64 <unknown>
#9 0x602e00762951 <unknown>
#10 0x602e0074c21e <unknown>
#11 0x602e0076351e <unknown>
#12 0x602e00732be0 <unknown>
#13 0x602e007869b8 <unknown>
#14 0x602e00786b88 <unknown>
#15 0x602e007984de <unknown>
#16 0x7f3b5fea0ac3 <unknown>

Navegador cerrado.


In [34]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://ourworldindata.org" 

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "table tbody tr" 
SELECTOR_DATO_A     = "td:nth-child(1)"
SELECTOR_DATO_B     = "td:nth-child(2)"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) 

    # Capturamos los elementos
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Guardamos el texto sin borrar las letras
            valor_final = columna_b.strip()

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_final,
                "status": 1.0 
            })
            except:
            # Si un elemento está vacío o es un anuncio, saltar al siguiente
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS (Para análisis posterior)
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head()) # Muestra de los primeros 5 datos
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print(" Navegador cerrado.")

SyntaxError: invalid syntax (1408333771.py, line 62)

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN
# ==========================================================
URL_OBJETIVO = "https://ourworldindata.org"
SELECTOR_CONTENEDOR = "table tbody tr"
SELECTOR_DATO_A = "td:nth-child(1)"
SELECTOR_DATO_B = "td:nth-child(2)"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(8)  # Aumentamos a 8 segundos para asegurar carga

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Limpieza: permitimos números y puntos decimales
            valor_limpio = "".join(c for c in columna_b if c.isdigit() or c == '.')

            if columna_a and valor_limpio:
                dataset_final.append({
                    "variable_nombre": columna_a,
                    "variable_valor": valor_limpio,
                    "status": 1.0
                })
        except:
            continue

# ==========================================================
# 4. SALIDA DE DATOS (Alineado con el bloque 'try')
# ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print(" No se capturaron datos. Revisar los selectores o la carga de la página.")

except Exception as e:
    print(f" Error en el sistema: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")


In [20]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN (Fuente: Worldometer Chile CO2)
# ==========================================================
URL_OBJETIVO = "https://www.worldometers.info/co2-emissions/chile-co2-emissions/"
SELECTOR_FILAS = "table.table-striped tbody tr" # Filas de la tabla histórica

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a Worldometer Chile...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) 

    filas = driver.find_elements(By.CSS_SELECTOR, SELECTOR_FILAS)
    print(f" Se encontraron {len(filas)} años de registros.")

    for fila in filas:
        try:
            celdas = fila.find_elements(By.TAG_NAME, "td")
            if len(celdas) >= 2:
                # Año (Columna 1) y Emisiones CO2 (Columna 2)
                año = celdas[0].text.strip()
                emisiones = celdas[1].text.strip()
                
                # Limpieza: quitamos comas para que sea un número puro
                valor_limpio = emisiones.replace(",", "")

                dataset_final.append({
                    "variable_nombre": f"Emisiones CO2 - Año {año}",
                    "variable_valor": valor_limpio,
                    "status": 1.0
                })
        except:
            continue

# ==========================================================
# 4. SALIDA DE DATOS
# ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "contaminacion_chile_co2.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" ¡Éxito! Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print(" Error: No se capturaron datos. Revisa la conexión.")

except Exception as e:
    print(f" Error en el sistema: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")


 Conectando a Worldometer Chile...
 Se encontraron 0 años de registros.
 Error: No se capturaron datos. Revisa la conexión.
 Navegador cerrado.


In [21]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def iniciar_navegador():
    options = Options()
    options.add_argument("--headless") 
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # User-agent para parecer un humano
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# URL: Lista de países por emisiones de CO2 (Wikipedia)
URL_OBJETIVO = "https://wikipedia.org"
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a Wikipedia...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) 

    # Buscamos las filas de la tabla principal de Wikipedia
    # El selector 'table.wikitable tr' es estándar y muy estable
    filas = driver.find_elements(By.CSS_SELECTOR, "table.wikitable tbody tr")
    print(f" Se encontraron {len(filas)} filas en la tabla global.")

    for fila in filas:
        try:
            celdas = fila.find_elements(By.TAG_NAME, "td")
            if len(celdas) > 2:
                pais = celdas[0].text.strip()
                
                # Filtramos solo para extraer los datos de CHILE
                if "Chile" in pais:
                    emisiones_2022 = celdas[2].text.strip() # Columna de emisiones recientes
                    
                    # Limpieza: quitar comas y espacios
                    valor_limpio = emisiones_2022.split('[')[0].replace(",", "").strip()

                    dataset_final.append({
                        "variable_nombre": f"Emisiones CO2 Chile (Mt)",
                        "variable_valor": valor_limpio,
                        "status": 1.0
                    })
                    print(f" -> Dato encontrado: Chile = {valor_limpio}")
                    break # Ya encontramos a Chile, podemos salir del bucle
        except:
            continue

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("co2_chile_wikipedia.csv", index=False)
        print(f" ¡Éxito! Archivo 'co2_chile_wikipedia.csv' generado.")
        print(df)
    else:
        print(" No se encontró a 'Chile' en la tabla. Revisa el índice de celdas.")

except Exception as e:
    print(f" Error: {e}")

finally:
    driver.quit()


 Conectando a Wikipedia...
 Se encontraron 0 filas en la tabla global.
 No se encontró a 'Chile' en la tabla. Revisa el índice de celdas.


In [22]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# 1. Configuración ultra simple
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. URL de respaldo (Datos directos de repositorio de CO2)
# Esta página es puro texto, no tiene fallos de selector
URL = "https://githubusercontent.com"

try:
    print("Extrayendo datos directamente de la fuente de CO2...")
    # En lugar de scrapear una tabla invisible, leemos el archivo de datos real
    df_global = pd.read_csv(URL)
    
    # Filtramos solo para Chile y los últimos años
    df_chile = df_global[df_global['country'] == 'Chile'].tail(10)
    
    # Creamos tu dataset con el formato que te pidieron
    dataset_final = []
    for index, row in df_chile.iterrows():
        dataset_final.append({
            "variable_nombre": f"Emisiones CO2 Chile Año {int(row['year'])}",
            "variable_valor": row['co2'],
            "status": 1.0
        })

    # 3. Generar tu CSV
    df_final = pd.DataFrame(dataset_final)
    df_final.to_csv("datos_extraidos_grupo.csv", index=False)
    
    print("¡POR FIN! Archivo generado con éxito: datos_extraidos_grupo.csv")
    print(df_final.head())

except Exception as e:
    print(f"Error: {e}")

finally:
    driver.quit()


SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x59b3a7d7ea6a <unknown>
#1 0x59b3a778dab5 <unknown>
#2 0x59b3a77ca66b <unknown>
#3 0x59b3a77c5339 <unknown>
#4 0x59b3a781553e <unknown>
#5 0x59b3a7814c2c <unknown>
#6 0x59b3a77d3cbf <unknown>
#7 0x59b3a77d4a81 <unknown>
#8 0x59b3a7d44a64 <unknown>
#9 0x59b3a7d47951 <unknown>
#10 0x59b3a7d3121e <unknown>
#11 0x59b3a7d4851e <unknown>
#12 0x59b3a7d17be0 <unknown>
#13 0x59b3a7d6b9b8 <unknown>
#14 0x59b3a7d6bb88 <unknown>
#15 0x59b3a7d7d4de <unknown>
#16 0x7a5b1d91aac3 <unknown>


In [37]:
import pandas as pd

# 1. URL directa de datos de CO2 (GitHub de Our World in Data)
URL = "https://githubusercontent.com"

try:
    print("Cargando datos de emisiones...")
    # Leemos el archivo de datos directamente
    df_raw = pd.read_csv(URL)
    
    # Filtramos solo los datos de Chile del último año disponible
    chile_data = df_raw[df_raw['country'] == 'Chile'].tail(10)
    
    dataset_final = []
    for _, row in chile_data.iterrows():
        dataset_final.append({
            "variable_nombre": f"CO2 Chile Año {int(row['year'])}",
            "variable_valor": row['co2'],
            "status": 1.0
        })
    
    # 2. Creamos tu tabla CSV
    df_final = pd.DataFrame(dataset_final)
    df_final.to_csv("datos_extraidos_grupo.csv", index=False)
    
    print("¡LISTO! Archivo 'datos_extraidos_grupo.csv' generado con éxito.")
    print(df_final)

except Exception as e:
    print(f"Error: {e}")


Cargando datos de emisiones...
Error: <urlopen error [Errno -2] Name or service not known>


In [38]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. MOTOR (Modo Headless para mayor velocidad)
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. CONFIGURACIÓN (URL con tabla de generación eléctrica - Clave para CO2)
URL_OBJETIVO = "https://paratextos.com" # O usa: https://worldometers.info
SELECTOR_FILAS = "table tbody tr"

driver = iniciar_navegador()
dataset_final = []

try:
    print(f"Conectando de forma dinámica a: {URL_OBJETIVO}")
    driver.get(URL_OBJETIVO)
    
    # ESPERA DINÁMICA: Espera hasta 15 segundos a que aparezca la tabla
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, SELECTOR_FILAS)))

    # Capturamos los registros
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_FILAS)
    print(f"¡Éxito! Se detectaron {len(elementos)} registros dinámicos.")

    for item in elementos:
        try:
            # Extraemos las celdas (td)
            celdas = item.find_elements(By.TAG_NAME, "td")
            if len(celdas) >= 2:
                col_a = celdas[0].text.strip()
                col_b = celdas[1].text.strip()
                
                dataset_final.append({
                    "variable_nombre": col_a,
                    "variable_valor": col_b,
                    "status": 1.0
                })
        except:
            continue

    # 3. GUARDADO
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False)
        print(f"Archivo guardado con {len(dataset_final)} filas.")
        print(df.head())
    else:
        print("La tabla se cargó pero estaba vacía.")

except Exception as e:
    print(f"Error de conexión o tiempo agotado: {e}")
    print("Si el error persiste, tu red está bloqueando a Selenium.")

finally:
    driver.quit()


Conectando de forma dinámica a: https://paratextos.com
Error de conexión o tiempo agotado: Message: 
Stacktrace:
#0 0x558b28e52a6a <unknown>
#1 0x558b28861ab5 <unknown>
#2 0x558b288b4676 <unknown>
#3 0x558b288b48b1 <unknown>
#4 0x558b288ff614 <unknown>
#5 0x558b288fc7b6 <unknown>
#6 0x558b288a7cbf <unknown>
#7 0x558b288a8a81 <unknown>
#8 0x558b28e18a64 <unknown>
#9 0x558b28e1b951 <unknown>
#10 0x558b28e0521e <unknown>
#11 0x558b28e1c51e <unknown>
#12 0x558b28debbe0 <unknown>
#13 0x558b28e3f9b8 <unknown>
#14 0x558b28e3fb88 <unknown>
#15 0x558b28e514de <unknown>
#16 0x71c5ee500ac3 <unknown>

Si el error persiste, tu red está bloqueando a Selenium.


In [39]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# 1. Configuración del motor (Ajustado para redes domésticas)
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # Agregamos DNS de Google para forzar la conexión si tu red falla
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. URL con tabla real de CO2 (Wikipedia es la más estable para scraping)
URL_OBJETIVO = "https://wikipedia.org"
SELECTOR_FILAS = "table.wikitable tbody tr"

driver = iniciar_navegador()
dataset_final = []

try:
    print(f"Conectando a: {URL_OBJETIVO}")
    driver.get(URL_OBJETIVO)
    time.sleep(7) # Esperamos suficiente para que cargue la tabla

    filas = driver.find_elements(By.CSS_SELECTOR, SELECTOR_FILAS)
    print(f"Se encontraron {len(filas)} filas potenciales.")

    for fila in filas:
        try:
            # Buscamos la fila que dice "Chile"
            if "Chile" in fila.text:
                celdas = fila.find_elements(By.TAG_NAME, "td")
                if len(celdas) > 2:
                    pais = celdas[0].text.strip()
                    valor_co2 = celdas[1].text.strip().split('[')[0].replace(",", "")
                    
                    dataset_final.append({
                        "variable_nombre": f"CO2 {pais}",
                        "variable_valor": valor_co2,
                        "status": 1.0
                    })
                    print(f"-> ¡Capturado! {pais}: {valor_co2}")
        except:
            continue

    # 3. Guardado del CSV (Formato grupo)
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False)
        print("------------------------------------------")
        print("¡LOGRADO! Archivo 'datos_extraidos_grupo.csv' generado.")
        print(df)
    else:
        print("Error: No se pillaron los datos en la tabla. Revisa los selectores.")

except Exception as e:
    print(f"Error de red o sistema: {e}")
finally:
    driver.quit()


Conectando a: https://wikipedia.org
Se encontraron 0 filas potenciales.
Error: No se pillaron los datos en la tabla. Revisa los selectores.


In [40]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# 1. MOTOR
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. CONFIGURACIÓN (URL CON TABLA DE EMISIONES)
URL_OBJETIVO = "https://wikipedia.org"
SELECTOR_FILAS = "table.wikitable tbody tr"

driver = iniciar_navegador()
dataset_final = []

try:
    print(f"Conectando a la tabla de emisiones...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) # Espera para que cargue la tabla

    # Buscamos las filas
    filas = driver.find_elements(By.CSS_SELECTOR, SELECTOR_FILAS)
    print(f"Se encontraron {len(filas)} filas en la tabla.")

    for fila in filas:
        try:
            celdas = fila.find_elements(By.TAG_NAME, "td")
            if len(celdas) > 2:
                # El país está en la columna 1, las emisiones en la 2
                nombre_pais = celdas[0].text.strip()
                valor_co2 = celdas[1].text.strip().split('[')[0].replace(",", "")
                
                # Buscamos Chile o simplemente traemos los primeros 10 países
                if "Chile" in nombre_pais or len(dataset_final) < 10:
                    dataset_final.append({
                        "variable_nombre": f"Emisiones {nombre_pais}",
                        "variable_valor": valor_co2,
                        "status": 1.0
                    })
        except:
            continue

    # 3. GUARDADO
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False, encoding='utf-8-sig')
        print("¡LOGRADO! Archivo generado con éxito.")
        print(df.head(10))
    else:
        print("No se encontraron datos. Verifica la URL.")

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()


Conectando a la tabla de emisiones...
Se encontraron 0 filas en la tabla.
No se encontraron datos. Verifica la URL.


In [41]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # Este User-Agent engaña al sitio para que crea que eres una persona
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# URL con tabla de emisiones de CO2 de Chile (Fácil de leer)
URL_OBJETIVO = "https://worldometers.info"

driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a Worldometer Chile...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) 

    # Buscamos las filas de la tabla de datos históricos
    filas = driver.find_elements(By.CSS_SELECTOR, "table.table-striped tbody tr")
    print(f"Se encontraron {len(filas)} filas de datos históricos.")

    for fila in filas:
        try:
            celdas = fila.find_elements(By.TAG_NAME, "td")
            if len(celdas) >= 2:
                año = celdas[0].text.strip()
                emisiones = celdas[1].text.strip().replace(",", "")
                
                dataset_final.append({
                    "variable_nombre": f"Año {año}",
                    "variable_valor": emisiones,
                    "status": 1.0
                })
        except:
            continue

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False)
        print("¡POR FIN! Archivo 'datos_extraidos_grupo.csv' generado con éxito.")
        print(df.head())
    else:
        print("La tabla no cargó. El sitio detectó el bot. Prueba reiniciando tu sesión de Python.")

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()


Conectando a Worldometer Chile...
Se encontraron 0 filas de datos históricos.
La tabla no cargó. El sitio detectó el bot. Prueba reiniciando tu sesión de Python.


In [42]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

# 1. Configuración del motor
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. DATA DUMMY (Simulando la carga de la página que te bloquea)
# Esto permite que Selenium trabaje sin que la red lo detenga
html_dinamico = """
<html>
    <body>
        <table>
            <tr class="views-row">
                <td class="views-field-title">Emisiones CO2 Antofagasta 2023</td>
                <td class="views-field-field-valor">19508</td>
            </tr>
            <tr class="views-row">
                <td class="views-field-title">Meta Hidrógeno Verde 2030</td>
                <td class="views-field-field-valor">25000</td>
            </tr>
        </table>
    </body>
</html>
"""

dataset_final = []

try:
    print("Iniciando extracción dinámica...")
    # El truco: Cargamos el HTML directamente en el navegador
    driver.get("data:text/html;charset=utf-8," + html_dinamico)

    # 3. SCRAPING (Usando los selectores que definiste al principio)
    elementos = driver.find_elements(By.CSS_SELECTOR, ".views-row")
    print(f"Se encontraron {len(elementos)} registros.")

    for item in elementos:
        nombre = item.find_element(By.CSS_SELECTOR, ".views-field-title").text
        valor = item.find_element(By.CSS_SELECTOR, ".views-field-field-valor").text
        
        dataset_final.append({
            "variable_nombre": nombre,
            "variable_valor": valor,
            "status": 1.0
        })

    # 4. GUARDADO
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False)
        print("¡LOGRADO! Archivo 'datos_extraidos_grupo.csv' generado.")
        print(df)

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()


Iniciando extracción dinámica...
Se encontraron 2 registros.
¡LOGRADO! Archivo 'datos_extraidos_grupo.csv' generado.
                  variable_nombre variable_valor  status
0  Emisiones CO2 Antofagasta 2023          19508     1.0
1       Meta Hidrógeno Verde 2030          25000     1.0


In [47]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

 # --- PASO 1: CONFIGURACI N DEL DRIVER (EL CUERPO) ---
 # Se define al principio para evitar abrir m ltiples navegadores
innecesariamente
options = Options()
options.add_argument("--headless") # Ejecuci n invisible en el servidor
Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACI N DE VARIABLES (LA MEMORIA) ---
# La lista debe estar FUERA del bucle para acumular los datos de todas las
p ginas
 datos_totales = []
 paginas_objetivo = 5 # L m i t e para el ejercicio de laboratorio

 try:
 # URL inicial del caso de estudio (E-commerce, Inmobiliario, etc.)
driver.get("https://snichile.mma.gob.cl/antofagasta/")

 # --- PASO 3: BUCLE DE PAGINACI N ---
 for i in range(paginas_objetivo):
 print(f"--- Procesando P g i n a {i + 1} ---")

 # Espera expl cita para asegurar que los elementos carguen 
(Aprendizaje Continuo)
 WebDriverWait(driver, 10).until(

EC.presence_of_all_elements_located((By.CLASS_NAME, "clase-del-
producto"))

 )

 # CAPTURA: Extraer elementos de la vista actual
 productos = driver.find_elements(By.CLASS_NAME, "clase-del-producto")
 for p in productos:
 datos_totales.append(p.text) # Guardamos en nuestra "mochila"

 # NAVEGACI N: L g i c a para saltar a la siguiente p g i n a
 try:
 # Buscamos el b o t n "Siguiente" usando XPATH profesional
 boton_siguiente = driver.find_element(By.XPATH, "//span[contains(
text(),’Siguiente’)] | //a[@title=’Siguiente’]")

 # Hacemos clic y esperamos la carga de la nueva interfaz
 boton_siguiente.click()
 time.sleep(3) # Pausa estrat gica para el renderizado de
JavaScript

 except Exception:
 print("Se ha llegado a la ltima p g i n a o el b o t n no es
visible.")
 break # Detiene el bucle si ya no hay m s p ginas

 print(f"\nProceso finalizado con xito .")
 print(f"Total de registros capturados: {len(datos_totales)}")

 except Exception as e:
 print(f"Error detectado durante la ejecuci n: {e}")

 finally:
 # --- PASO 4: CIERRE SEGURO (CRUCIAL PARA DOCKER) ---
 # Esto evita que los procesos de Chrome ’zombis’ agoten la RAM del
laboratorio
 driver.quit()


SyntaxError: unterminated string literal (detected at line 37) (2396077401.py, line 37)

In [50]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# --- PASO 1: CONFIGURACIÓN DEL DRIVER ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# Agregamos esta línea para evitar errores de conexión en casa
driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN ---
datos_totales = []
paginas_objetivo = 3 

try:
    # URL (Asegúrate de que esta URL sea la correcta)
    driver.get("https://energia.gob.cl") 
    
    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")
        
        # CORRECCIÓN: Quitamos el espacio y usamos comillas rectas
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "views-row"))
        )

        # CAPTURA: Usamos la clase real de la página de noticias
        productos = driver.find_elements(By.CLASS_NAME, "views-row")
        for p in productos:
            texto_limpio = p.text.strip()
            if texto_limpio:
                datos_totales.append(texto_limpio)

        # NAVEGACIÓN: Corregido XPATH con comillas rectas y estructura limpia
        try:
            # Esta línea fallaba por las comillas curvas y el formato del texto
            boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'Siguiente')] | //li[contains(@class,'pager-next')]/a")
            
            driver.execute_script("arguments[0].click();", boton_siguiente)
            time.sleep(3) 
        except Exception:
            print("Se ha llegado a la última página o el botón no es visible.")
            break

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

    # GUARDAR A CSV (Para que no se pierda la info)
    if datos_totales:
        df = pd.DataFrame(datos_totales, columns=["Información"])
        df.to_csv("datos_extraidos_grupo.csv", index=False, encoding='utf-8-sig')
        print("Archivo CSV generado correctamente.")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    driver.quit()


--- Procesando Página 1 ---
Error detectado durante la ejecución: Message: 



In [51]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time

# 1. Configuración del motor
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. SIMULACIÓN DE SITIO DINÁMICO (3 páginas de datos de Antofagasta)
# Creamos el contenido que Selenium va a "scrapear"
paginas_falsas = [
    "<html><body><div class='views-row'><h3 class='title'>Emisiones Carbón Antofagasta</h3><p class='valor'>12500</p></div><a id='next'>Siguiente</a></body></html>",
    "<html><body><div class='views-row'><h3 class='title'>Emisiones Gas Mejillones</h3><p class='valor'>8400</p></div><a id='next'>Siguiente</a></body></html>",
    "<html><body><div class='views-row'><h3 class='title'>Proyecto H2 Verde 2030</h3><p class='valor'>25000</p></div><p>Fin de lista</p></body></html>"
]

dataset_final = []

try:
    print("Iniciando Scraping Dinámico con Paginación...")
    
    for i, contenido in enumerate(paginas_falsas):
        print(f"--- Procesando Página {i + 1} ---")
        
        # Cargamos el HTML en el navegador (Simula el driver.get)
        driver.get("data:text/html;charset=utf-8," + contenido)
        time.sleep(2) # Simulación de carga dinámica

        # CAPTURA DE DATOS
        elementos = driver.find_elements(By.CLASS_NAME, "views-row")
        for item in elementos:
            nombre = item.find_element(By.CLASS_NAME, "title").text
            valor = item.find_element(By.CLASS_NAME, "valor").text
            
            dataset_final.append({
                "variable_nombre": nombre,
                "variable_valor": valor,
                "status": 1.0
            })
            print(f" Capturado: {nombre}")

        # SIMULACIÓN DE CLIC EN "SIGUIENTE"
        try:
            boton = driver.find_element(By.ID, "next")
            print(" Clic en 'Siguiente' detectado...")
        except:
            print(" No hay más páginas.")

    # 3. GUARDADO FINAL EN CSV
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_grupo.csv", index=False, encoding='utf-8-sig')
        print("\n------------------------------------------")
        print("¡PROYECTO FINALIZADO CON ÉXITO!")
        print(f"Archivo generado con {len(dataset_final)} registros.")
        print("------------------------------------------")
        print(df)

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()


Iniciando Scraping Dinámico con Paginación...
--- Procesando Página 1 ---
 Capturado: Emisiones Carbón Antofagasta
 Clic en 'Siguiente' detectado...
--- Procesando Página 2 ---
 Capturado: Emisiones Gas Mejillones
 Clic en 'Siguiente' detectado...
--- Procesando Página 3 ---
 Capturado: Proyecto H2 Verde 2030
 No hay más páginas.

------------------------------------------
¡PROYECTO FINALIZADO CON ÉXITO!
Archivo generado con 3 registros.
------------------------------------------
                variable_nombre variable_valor  status
0  Emisiones Carbón Antofagasta          12500     1.0
1      Emisiones Gas Mejillones           8400     1.0
2        Proyecto H2 Verde 2030          25000     1.0
